In [1]:
!pip install simpy
import simpy
import random
import statistics
import networkx as nx
import matplotlib.pyplot as plt
import copy
from contextlib import redirect_stdout

In [15]:
import random
import simpy
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import sys
import pandas as pd


# 重定向输出到文件
original_stdout = sys.stdout
#sys.stdout = open("simulation_output.txt", "w")
sys.stdout = original_stdout

TOTAL_PIECES = 10  # 总共的文件块数 / Total number of file pieces
SIMULATION_TIME = 100  # 模拟的总时长 / Total simulation time

class Strategy:
    @staticmethod
    def equal_distribution_strategy(peer, target_peer_id):
        """根据当前的请求数，均匀分配上传带宽 / Evenly distribute upload bandwidth based on current number of requests"""
        number_of_requests = peer.current_requests
        available_bandwidth_per_request = peer.upload_speed / max(1, number_of_requests)  # 防止除0
        return available_bandwidth_per_request

    @staticmethod
    def tit_for_tat(peer, target_peer_id):
        """根据对方从当前用户得到的总下载量占比分配带宽 / Allocate bandwidth based on the ratio of what the target peer has uploaded"""
        total_downloaded = peer.total_downloaded
        if total_downloaded == 0:
            # 如果当前用户还没有下载任何东西，则按平等策略分配
            return Strategy.equal_distribution_strategy(peer, target_peer_id)

        # 获取目标 peer 从当前 peer 这里下载的总量
        download_from_peer = peer.upload_history.get(target_peer_id, 0)
        # 按照 tit-for-tat 策略计算带宽占比
        bandwidth_ratio = download_from_peer / total_downloaded
        available_bandwidth = bandwidth_ratio * peer.upload_speed
        return available_bandwidth

    def random_bandwidth_distribution(peer):
        """随机生成带宽分配并保存到 bandwidth_allocation 中 / Randomly generate bandwidth allocation and save it"""
        total_bandwidth = peer.upload_speed
        number_of_peers = len(peer.peers)
        allocations = {}

        # 随机分配带宽给每个请求的 peer
        for p in peer.peers:
            allocation = random.uniform(0, total_bandwidth / max(1, number_of_peers))
            allocations[p.id] = allocation
            total_bandwidth -= allocation

        # 保存分配到 peer 的 bandwidth_allocation 中
        peer.bandwidth_allocation = allocations

    @staticmethod
    def allocate_bandwidth(peer, target_peer_id):
        """根据带宽分配表分配带宽，如果表为空则随机分配 / Allocate bandwidth based on existing allocation, or randomly if none exists"""
        if not peer.bandwidth_allocation:
            Strategy.random_bandwidth_distribution(peer)

        # 按照分配表分配带宽
        return peer.bandwidth_allocation.get(target_peer_id, 0)

    def demand_priority(peer, target_peer_id):
        """
        优先上传需求最小的 peer 的策略 /
        Prioritize peers with the smallest demand
        """
        total_requests = sum(len(request_list) for request_list in peer.request_count_dict.values())

        if total_requests == 0:
            # 如果没有请求，则平均分配带宽
            return peer.upload_speed / len(peer.peers)

        # 计算目标 peer 的需求权重（需求越小，权重越高）
        target_requests = len(peer.request_count_dict.get(target_peer_id, []))
        if target_requests == 0:
            # 如果目标 peer 没有请求过文件，分配最低优先级的带宽
            return peer.upload_speed / len(peer.peers)

        demand_ratio = (1 / target_requests) / sum(
          1 / len(request_list) for request_list in peer.request_count_dict.values() if len(request_list) > 0
      )
        # 分配的带宽比例
        allocated_bandwidth = demand_ratio * peer.upload_speed
        return allocated_bandwidth

    def free_rider_strategy(peer, target_peer_id):
        """不上传任何东西 / Do not upload anything (Free Rider strategy)"""
        return 0

class Peer:
    def __init__(self, env, id, upload_speed, download_speed, animation, strategy):
        self.env = env
        self.id = id
        self.upload_speed = upload_speed  # 上传带宽 / Upload bandwidth
        self.download_speed = download_speed  # 下载带宽 / Download bandwidth
        self.file_pieces = set()  # 当前拥有的文件块 / File pieces currently owned
        self.peers = []  # 其他对等节点 / Other peers
        self.upload_resource = simpy.Resource(env, capacity=upload_speed)  # 上传带宽作为资源 / Upload bandwidth as a resource
        self.current_requests = 0  # 当前的请求数 / Current number of requests
        self.upload_history = {}  # 记录上传历史 / Dictionary to track upload history
        self.total_downloaded = 0  # 记录总共下载的文件大小 / Total downloaded file size
        self.bandwidth_allocation = {}  # 带宽分配，记录对每个 peer 的分配情况 / Dictionary to store bandwidth allocation
        self.download_times = {}
        self.request_count_dict = {}
        self.env.process(self.run())
        self.animation = animation  # 引入 Animation 对象
        self.strategy = strategy  # 当前 peer 的带宽分配策略

    def add_peer(self, peer):
        self.peers.append(peer)

    def request_piece(self, piece, target):
        """请求特定文件块并动态调整带宽 / Request a specific file piece and dynamically adjust bandwidth"""
        if piece in target.file_pieces:
            #print(f'Time {self.env.now}: Peer {self.id} requests file piece {piece} from Peer {target.id}')
            request_log.append({
                'time': self.env.now,
                'from_peer': self.id,
                'to_peer': target.id,
                'piece': piece
            })

            if self.id not in target.request_count_dict:
                    target.request_count_dict[self.id] = []

                # 将请求添加到目标Peer的请求列表
            target.request_count_dict[self.id].append(piece)
            while target.request_count_dict[self.id][0] != piece:
              yield self.env.timeout(0.1)

            with target.upload_resource.request() as req:
                target.current_requests += 1
                piece_size = Peer.piece_size
                downloaded = 0 # 初始化已下载部分 / Initialize downloaded portion

                while downloaded < piece_size:
                    # 根据 target peer 的策略分配带宽
                    if target.strategy == "tit_for_tat":
                        available_bandwidth_per_request = Strategy.tit_for_tat(target, self.id)
                    elif target.strategy == "random_bandwidth":
                        available_bandwidth_per_request = Strategy.allocate_bandwidth(target, self.id)
                    elif target.strategy == "demand_priority":
                        available_bandwidth_per_request = Strategy.demand_priority(target, self.id)
                    elif target.strategy == "free_rider_strategy":
                        available_bandwidth_per_request = Strategy.free_rider_strategy(target, self.id)
                    else:
                        available_bandwidth_per_request = Strategy.equal_distribution_strategy(target, self.id)

                    data_per_step = available_bandwidth_per_request * 0.1  # 假设时间步为 0.1 秒

                    downloaded += data_per_step
                    yield self.env.timeout(0.1)

                    # 计算进度百分比
                    progress_percentage = (downloaded / piece_size) * 100
                    #print(f'Time {self.env.now:.1f}: Peer {self.id} has downloaded {progress_percentage:.1f}% of file piece {piece}')

                    # 调用动画更新进度
                    if self.animation:
                      self.animation.update_progress((target.id, self.id), round(progress_percentage, 2))

                target.current_requests -= 1
                target.request_count_dict[self.id].remove(piece)
                #print(f'Time {self.env.now}: Peer {self.id} completed download of piece {piece} from Peer {target.id}')
                self.file_pieces.add(piece)

                self.download_times[piece] = self.env.now
                #print(f'Time {self.env.now:.1f}: Peer {self.id} completed download of file piece {piece}')

                # 记录 target 传输给当前 peer 的总文件大小
                if target.id not in self.upload_history:
                    self.upload_history[target.id] = 0
                self.upload_history[target.id] += piece_size  # 将传输的文件块大小加入历史记录

                # 更新总下载文件大小
                self.total_downloaded += piece_size
                #print(f"Peer {self.id} received {piece_size} MB from Peer {target.id}. Total received: {self.upload_history[target.id]} MB")
                #print(f"Peer {self.id} has downloaded a total of {self.total_downloaded} MB")

        else:
            print(f"Time {self.env.now:.1f}: Peer {self.id} found that Peer {target.id} no longer has file piece {piece}")


    def run(self):
        while True:
            if len(self.file_pieces) == TOTAL_PIECES:
                break
            yield self.env.timeout(1)

            missing_pieces = set(range(TOTAL_PIECES)) - self.file_pieces
            if not missing_pieces:
                break

            request_tasks = []
            requesting_pieces = set()

            for piece in missing_pieces:
                if piece in requesting_pieces:
                    continue
                shuffled_peers = self.peers[:]
                random.shuffle(shuffled_peers)

                for target_peer in shuffled_peers:
                    if piece in target_peer.file_pieces:
                        # 在找到目标节点后立即连接，并开始请求
                        if self.animation:
                          self.animation.connect_peers(target_peer.id, self.id)
                        request_tasks.append(self.env.process(self.request_piece(piece, target_peer)))
                        requesting_pieces.add(piece)
                        break

            if request_tasks:
                yield self.env.all_of(request_tasks)

class Animation:
    def __init__(self, num_peers):
        self.num_peers = num_peers
        self.G = nx.DiGraph()
        self.pos = None
        self.edge_labels = {}
        self.fig, self.ax = plt.subplots()  # 创建画布
        self.updates = []  # 用于存储每一帧的更新（边和进度）
        self.frame_count = 0  # 记录调用 update_progress 的次数

    def generate_peers(self):
        """生成Peer节点"""
        self.G.add_node(0, label='Peer 0')
        for i in range(1, self.num_peers):
            self.G.add_node(i, label=f'Peer {i}')
        self.pos = nx.circular_layout(self.G)

    def connect_peers(self, from_peer, to_peer):
        """为成功请求的两个Peer之间连线"""
        if not self.G.has_edge(from_peer, to_peer):
            self.G.add_edge(from_peer, to_peer, progress=0)  # 为两个成功请求的节点添加一条边
        self.edge_labels[(from_peer, to_peer)] = '0%'

    def update_progress(self, edge, progress):
        """更新指定边的进度，并在动画中展示"""
        from_peer, to_peer = edge
        if self.G.has_edge(from_peer, to_peer):
            self.G[from_peer][to_peer]['progress'] = progress
            self.edge_labels[(from_peer, to_peer)] = f'{progress}%'

        # 每隔一定更新次数记录一次帧
        if self.frame_count % 1 == 0:  # 跳过一半的帧
            self.updates.append((edge, progress))
        self.frame_count += 1

    def animate(self, i):
        """逐帧更新图像的动画函数"""
        self.ax.clear()

        # 绘制节点和边
        nx.draw(self.G, self.pos, with_labels=True, labels=nx.get_node_attributes(self.G, 'label'),
                node_color='lightblue', node_size=1000, ax=self.ax)
        nx.draw_networkx_edges(self.G, self.pos, ax=self.ax)

        # 逐帧应用进度更新
        if i < len(self.updates):
            edge, progress = self.updates[i]
            from_peer, to_peer = edge
            self.edge_labels[(from_peer, to_peer)] = f'{progress}%'

        # 绘制边上的进度标签
        nx.draw_networkx_edge_labels(self.G, self.pos, edge_labels=self.edge_labels, ax=self.ax)

    def save_animation(self, filename='animation.mp4', target_duration=100):

      effective_frames = len(self.updates)  # 实际记录的有效帧数
      if effective_frames > 0:
          # 动态计算帧间隔，确保总时长不超过 target_duration
          interval = (target_duration * 1000) / effective_frames  # 单位：毫秒
          interval = max(1, interval)  # 确保帧间隔不低于 20 毫秒
      else:
          interval = 100  # 默认间隔

      # 使用有效帧数作为 frames
      anim = FuncAnimation(self.fig, self.animate, frames=effective_frames, interval=interval, repeat=False)
      anim.save(filename, writer='ffmpeg')
      print(f"Animation saved as {filename} with {effective_frames} frames and interval {interval:.1f} ms.")

request_log = []

def run_simulation(num_peers, enable_animation=True):
    env = simpy.Environment()
    peers = []
    animation = None  # 创建动画实例
    piece_download_matrix = [["" for _ in range(TOTAL_PIECES)] for _ in range(num_peers)]
    time_matrix = [["——" for _ in range(TOTAL_PIECES)] for _ in range(num_peers)]

    if enable_animation:
        animation = Animation(num_peers)  # 创建动画实例
        animation.generate_peers()

    Peer.file_size = random.uniform(10, 20)  # 假设文件大小在10MB到20MB之间
    Peer.piece_size = Peer.file_size / TOTAL_PIECES

    # 创建多个对等节点
    print("Peer Initial Values:")
    print("{:<10}{:<15}{:<15}{:<25}".format("Peer ID", "Upload BW", "Download BW", "Strategy"))
    for i in range(num_peers):
        upload_speed = random.randint(1, 5)
        download_speed = random.randint(30, 50)
        strategy = random.choice(
            ["random_bandwidth", "tit_for_tat", "equal_distribution_strategy", "demand_priority", "free_rider_strategy"]
        )
        peer = Peer(env, i, upload_speed, download_speed, animation, strategy)
        peers.append(peer)
        print("{:<10}{:<15.2f}{:<15.2f}{:<25}".format(i, upload_speed, download_speed, strategy))

        num_pieces = random.randint(0, TOTAL_PIECES - 1)
        peer.file_pieces = set(random.sample(range(TOTAL_PIECES), num_pieces))

    all_pieces = list(range(TOTAL_PIECES))  # 文件块的编号
    random.shuffle(all_pieces)
    for index, piece in enumerate(all_pieces):
        peer_id = index % num_peers
        peers[peer_id].file_pieces.add(piece)

    # 连接所有对等节点
    for peer in peers:
        for other_peer in peers:
            if peer != other_peer:
                peer.add_peer(other_peer)

    # 运行模拟
    env.run(until=SIMULATION_TIME)

    print("\nDownload Times:")
    for peer in peers:
        print(f"Peer {peer.id}:")
        for piece, time in sorted(peer.download_times.items()):
            print(f"  File piece {piece}: completed at time {time:.1f}")

    for log in request_log:
        from_peer = log["from_peer"]
        to_peer = log["to_peer"]
        piece = log["piece"]
        piece_download_matrix[from_peer][piece] = f"{piece}.Peer{to_peer}"

    # 填充时间矩阵
    for peer in peers:
        for piece, time in sorted(peer.download_times.items()):
            time_matrix[peer.id][piece] = f"{time:.1f}"
        for log in request_log:
            if log["from_peer"] == peer.id and time_matrix[peer.id][log["piece"]] == "——":
                time_matrix[peer.id][log["piece"]] = "non-finished"

    # 计算每个Peer的总完成时间
    total_finished_time = []
    for peer in peers:
        finished_times = [time for time in peer.download_times.values()]
        total_time = max(finished_times) if finished_times else "non-finished"
        total_finished_time.append(total_time)

    # 输出 Piece Download Matrix
    print("\nPiece Download Matrix:")
    print(pd.DataFrame(
        piece_download_matrix,
        index=[f"Peer {i}" for i in range(num_peers)],
        columns=[f"Piece {i}" for i in range(TOTAL_PIECES)]
    ).to_string(index=True, line_width=1000))

    # 输出 Time Matrix
    print("\nTime Matrix:")
    print(pd.DataFrame(
        time_matrix,
        index=[f"Peer {i}" for i in range(num_peers)],
        columns=[f"Piece {i}" for i in range(TOTAL_PIECES)]
    ).to_string(index=True, line_width=1000))

    # 输出 Total Finished Time
    print("\nTotal Finished Time for Each Peer:")
    for i, peer in enumerate(peers):
        if any(time_matrix[i][j] == "non-finished" for j in range(TOTAL_PIECES)):
            # 如果该 Peer 的时间矩阵中有 "non-finished"，打印 "non-finished"
            print(f"Peer {i}: non-finished")
        else:
            # 如果没有 "non-finished"，计算总完成时间并打印保留一位小数
            finished_times = [time for time in peer.download_times.values()]
            total_time = max(finished_times) if finished_times else "non-finished"
            print(f"Peer {i}: {total_time:.1f}")

    print("\n")
    for peer in peers:
        if peer.download_times:
            avg_time = sum(peer.download_times.values()) / len(peer.download_times)
            print(f"Peer {peer.id}: Average download time = {avg_time:.2f}")
        else:
            print(f"Peer {peer.id}: No pieces downloaded.")

    # 保存动画为 mp4
    if enable_animation:
        target_video_duration = 60
        animation.save_animation(filename="animation.mp4", target_duration=target_video_duration)
        print(f"Animation saved as animation.mp4 with {animation.frame_count} frames.")
    else:
        print("Animation generation is disabled.")


def calculate_strategy_averages(num_simulations, num_peers):
    """
    运行多次模拟并统计每种策略的平均时间。

    参数:
    - num_simulations: 总共运行的模拟次数
    - num_peers: 每次模拟的节点数量

    返回:
    - 各策略的最终平均下载时间
    """
    strategy_results = {
        "random_bandwidth": [],
        "tit_for_tat": [],
        "equal_distribution_strategy": [],
        "demand_priority": [],
        "free_rider_strategy": []
    }

    # 第一次创建 Peers 并保存初始属性
    env = simpy.Environment()
    peers = create_peers(env, num_peers, is_first_time=True)

    # 后续多次模拟
    for _ in range(num_simulations):
        env = simpy.Environment()
        # 复用初始的 Peers 属性（is_first_time=False）
        peers = create_peers(env, num_peers, is_first_time=False)
        env.run(until=SIMULATION_TIME)

        # 统计每种策略的平均时间
        for peer in peers:
            if peer.download_times:  # 如果该 Peer 下载了文件块
                avg_time = sum(peer.download_times.values()) / len(peer.download_times)
                strategy_results[peer.strategy].append(avg_time)

    # 计算每种策略的总体平均时间
    final_averages = {}
    for strategy, times in strategy_results.items():
        if times:
            final_averages[strategy] = sum(times) / len(times)
        else:
            final_averages[strategy] = None

    return final_averages



def create_peers(env, num_peers, is_first_time):
    """
    创建多个对等节点并初始化其属性。
    """
    global saved_peer_attributes
    peers = []

    # 确保文件大小和块大小一致
    Peer.file_size = random.uniform(10, 20)  # 假设文件大小在10MB到20MB之间
    Peer.piece_size = Peer.file_size / TOTAL_PIECES

    if is_first_time:
        saved_peer_attributes = []  # 清空保存的属性列表

        # 保存初始状态
        for i in range(num_peers):
            upload_speed = random.randint(1, 5)
            download_speed = random.randint(30, 50)
            num_pieces = random.randint(0, TOTAL_PIECES - 1)
            initial_pieces = set(range(TOTAL_PIECES)) if i == -1 else set(random.sample(range(TOTAL_PIECES), num_pieces))
            initial_pieces.add(i % TOTAL_PIECES)
            saved_peer_attributes.append({
                "upload_speed": upload_speed,
                "download_speed": download_speed,
                "initial_pieces": initial_pieces
            })

    else:
        # 从保存的属性中加载
        for i, peer_attr in enumerate(saved_peer_attributes):
            upload_speed = peer_attr["upload_speed"]
            download_speed = peer_attr["download_speed"]
            initial_pieces = peer_attr["initial_pieces"]

            # **重新创建 Peer 对象**
            strategy = random.choice(["random_bandwidth", "tit_for_tat", "equal_distribution_strategy", "demand_priority","free_rider_strategy"])
            peer = Peer(env, i, upload_speed, download_speed, None, strategy)

            # 重新分配属性
            peer.file_pieces = set(initial_pieces)  # 注意使用 set() 防止引用问题
            peers.append(peer)

    # 连接所有对等节点
    for peer in peers:
        for other_peer in peers:
            if peer != other_peer:
                peer.add_peer(other_peer)

    return peers


final_averages = calculate_strategy_averages(num_simulations=300, num_peers=10)

print("Final Average Download Times for Each Strategy:")
for strategy, avg_time in final_averages.items():
    print(f"{strategy}: {avg_time:.2f} units")




Final Average Download Times for Each Strategy:
random_bandwidth: 7.70 units
tit_for_tat: 8.12 units
equal_distribution_strategy: 9.65 units
demand_priority: 8.24 units
free_rider_strategy: 11.16 units


In [4]:
run_simulation(5, False)


In [ ]:
final_averages = calculate_strategy_averages(num_simulations=200, num_peers=10)

print("Final Average Download Times for Each Strategy:")
for strategy, avg_time in final_averages.items():
    print(f"{strategy}: {avg_time:.2f} units")